#1. Set-Up
---

First lets set up the environment, import all libraries used, set Kaggle API token and download the "parking" dataset.

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier


!pip install -q --upgrade kaggle

os.environ["KAGGLE_API_TOKEN"] = "KGAT_5321fd78f78cbd590b9dd1c31bbca744"

!kaggle datasets download -d ddsshubham/cnrpark-ext
!unzip -q cnrpark-ext.zip -d cnrpark_ext_dataset
!ls cnrpark_ext_dataset


Dataset URL: https://www.kaggle.com/datasets/ddsshubham/cnrpark-ext
License(s): unknown
100% 1.49G/1.49G [00:24<00:00, 64.3MB/s]

CNR-EXT_FULL_IMAGE_1000x750  CNRParkEXT.csv
CNR-EXT-Patches-150x150      CNRPark-Patches-150x150


In [ ]:
!find cnrpark_ext_dataset/CNR-EXT-Patches-150x150 -maxdepth 2 -type d | head -50

!find cnrpark_ext_dataset/CNRPark-Patches-150x150 -maxdepth 2 -type d | head -50

cnrpark_ext_dataset/CNR-EXT-Patches-150x150
cnrpark_ext_dataset/CNR-EXT-Patches-150x150/LABELS
cnrpark_ext_dataset/CNR-EXT-Patches-150x150/PATCHES
cnrpark_ext_dataset/CNR-EXT-Patches-150x150/PATCHES/SUNNY
cnrpark_ext_dataset/CNR-EXT-Patches-150x150/PATCHES/RAINY
cnrpark_ext_dataset/CNR-EXT-Patches-150x150/PATCHES/OVERCAST
cnrpark_ext_dataset/CNRPark-Patches-150x150
cnrpark_ext_dataset/CNRPark-Patches-150x150/B
cnrpark_ext_dataset/CNRPark-Patches-150x150/B/busy
cnrpark_ext_dataset/CNRPark-Patches-150x150/B/free
cnrpark_ext_dataset/CNRPark-Patches-150x150/A
cnrpark_ext_dataset/CNRPark-Patches-150x150/A/busy
cnrpark_ext_dataset/CNRPark-Patches-150x150/A/free


#2. Paths to Use and Sample Images
---
Now we are setting up the paths to use inside the dataset folder, as well as confirming the amount of samples available for training.

We also show a few sample images from the dataset, to confirm that images are loading correclty and that the classes make sense.

In [ ]:
BASE_PATH = "/content/cnrpark_ext_dataset/CNRPark-Patches-150x150"

A_FREE_PATH = os.path.join(BASE_PATH, "A", "free")
A_BUSY_PATH = os.path.join(BASE_PATH, "A", "busy")
B_FREE_PATH = os.path.join(BASE_PATH, "B", "free")
B_BUSY_PATH = os.path.join(BASE_PATH, "B", "busy")

print("A free:", A_FREE_PATH)
print("A busy:", A_BUSY_PATH)
print("B free:", B_FREE_PATH)
print("B busy:", B_BUSY_PATH)

print()

print("A free count:", len(os.listdir(A_FREE_PATH)))
print("A busy count:", len(os.listdir(A_BUSY_PATH)))
print("B free count:", len(os.listdir(B_FREE_PATH)))
print("B busy count:", len(os.listdir(B_BUSY_PATH)))

print()

A free: /content/cnrpark_ext_dataset/CNRPark-Patches-150x150/A/free
A busy: /content/cnrpark_ext_dataset/CNRPark-Patches-150x150/A/busy
B free: /content/cnrpark_ext_dataset/CNRPark-Patches-150x150/B/free
B busy: /content/cnrpark_ext_dataset/CNRPark-Patches-150x150/B/busy

A free count: 2550
A busy count: 3621
B free count: 1632
B busy count: 4781



In [ ]:
def show_sample_images(folder_path, title, num_images = 5):
  files = os.listdir(folder_path)[:num_images]

  plt.figure(figsize=(15, 3))
  for i, filename in enumerate(files):
    file_path = os.path.join(folder_path, filename)
    img = cv2.imread(file_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    plt.subplot(1, num_images, i + 1)
    plt.imshow(img)
    plt.title(title)
    plt.axis('off')

  plt.show()

print("A free count:", len(os.listdir(A_FREE_PATH)))
print("A busy count:", len(os.listdir(A_BUSY_PATH)))
print("---------------------")
print("B free count:", len(os.listdir(B_FREE_PATH)))
print("B busy count:", len(os.listdir(B_BUSY_PATH)))


A free count: 2550
A busy count: 3621
B free count: 1632
B busy count: 4781


# 3. Dataset - Load, Preprocess, Build and Split
---
First we load and preprocess the dataset, so that each image becomes a number 0 or 1.

`image → grayscale → resize → flatten → feature vector`


Random Forest and Support Vector Machines do not take images, they need a numeric input.

#
Our next step is to create features `X` and labels `y`, so that we have a parking spot image turned into numbers `X[1]`, and its answer `y[i]`: 0 empty or 1 occupied.

#
Then we split the dataset into **80% training** and **20% testing**; this is how we check if the model actually learned and works.



In [ ]:
IMG_SiZE = (32, 32)

def load_images_from_folder(folder_path, label):
  data = []
  labels = []

  for filename in os.listdir(folder_path):
    file_path = os.path.join(folder_path, filename)
    img = cv2.imread(file_path)

    if img is None:
      continue

    img = cv2.resize(img, IMG_SiZE)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    ftrs = img.flatten()

    data.append(ftrs)
    labels.append(label)

  return data, labels

In [ ]:
free_a_data, free_a_labels = load_images_from_folder(A_FREE_PATH, 0)
busy_a_data, busy_a_labels = load_images_from_folder(A_BUSY_PATH, 1)

free_b_data, free_b_labels = load_images_from_folder(B_FREE_PATH, 0)
busy_b_data, busy_b_labels = load_images_from_folder(B_BUSY_PATH, 1)

X = np.array(free_a_data + busy_a_data + free_b_data + busy_b_data)
y = np.array(free_a_labels + busy_a_labels + free_b_labels + busy_b_labels)

print("---------------------")

print("Samples:", len(X))
print("Feature vector size:", X.shape[1])
print("Labels shape:", y.shape)

print("---------------------")

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

print("---------------------")

---------------------
Samples: 12584
Feature vector size: 1024
Labels shape: (12584,)
---------------------
Training samples: 10067
Testing samples: 2517
---------------------


#4. Training
---

### Random Forest Model
Creation of the Random Forest Model with 100 decision trees, each tree votes on wheter the image is empty or occupied.

Then training, so the model will look at training examples and learn patters that will separate **empty spots** and **occupied spots**.

Lastly predicts on test images it has never seen. Metrics used:
* Accuracy
* Precision, recall, F1 Score
* Confusion Matrix

### Support Vector Machine
Creation of the SVM Model, so SVM tries to find the best boundary between, **empty parking spots** and **occupied parking spots**. We used a linear kernel.




In [17]:
import joblib

random_forest_model = RandomForestClassifier(n_estimators=100, random_state=42)
random_forest_model.fit(X_train, y_train)

random_forest_predictions = random_forest_model.predict(X_test)

print("Random Forest Accuracy: ", accuracy_score(y_test, random_forest_predictions))
print("\nClassification Report: \n")
print(classification_report(y_test, random_forest_predictions))
print("\nConfusion Matrix: \n")
print(confusion_matrix(y_test, random_forest_predictions))

svm_model = SVC(kernel='linear', random_state=42)
svm_model.fit(X_train, y_train)

print()
print("--------------------------------------------------------")
print()

svm_predictions = svm_model.predict(X_test)

print("SVM Accuracy: ", accuracy_score(y_test, svm_predictions))
print("\nClassification Report: \n")
print(classification_report(y_test, svm_predictions))
print("\nConfusion Matrix: \n")
print(confusion_matrix(y_test, svm_predictions))

joblib.dump(random_forest_model, 'random_forest_parking_model.pkl')
joblib.dump(svm_model, 'svm_parking_model.pkl')

print()
print("--------------------------------------------------------")
print()

print("Models successfully saved.")


Random Forest Accuracy:  0.9884783472387764

Classification Report: 

              precision    recall  f1-score   support

           0       0.99      0.97      0.98       836
           1       0.99      1.00      0.99      1681

    accuracy                           0.99      2517
   macro avg       0.99      0.98      0.99      2517
weighted avg       0.99      0.99      0.99      2517


Confusion Matrix: 

[[ 812   24]
 [   5 1676]]

--------------------------------------------------------

SVM Accuracy:  0.9356376638855781

Classification Report: 

              precision    recall  f1-score   support

           0       0.88      0.93      0.91       836
           1       0.97      0.94      0.95      1681

    accuracy                           0.94      2517
   macro avg       0.92      0.94      0.93      2517
weighted avg       0.94      0.94      0.94      2517


Confusion Matrix: 

[[ 781   55]
 [ 107 1574]]

--------------------------------------------------------

Mo